In [1]:
!nvidia-smi

Thu Oct 30 12:07:49 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           On  |   00000000:5E:00.0 Off |                    0 |
| N/A   34C    P0             26W /  250W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !pip install transformers peft datasets protobuf tiktoken sentencepiece

In [2]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

In [3]:
# Configure quantization (4-bit) for temporary experimentation
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model with quantization
# TODO: try to use the full-precision model once this pipeline is ready.
model_name = "openlm-research/open_llama_3b_v2"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

In [4]:
tokenizer = AutoTokenizer.from_pretrained( "openlm-research/open_llama_3b_v2", use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # Explicitly set padding side TODO: better understand this.

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


In [5]:
# LoRA fine-tuning
model = prepare_model_for_kbit_training(model)

# TODO: experiment with htis.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # explore which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,324,800 || all params: 3,431,798,400 || trainable%: 0.1552


In [6]:
# Prepare dataset
dataset = load_dataset("gtfintechlab/fomc_communication")

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)


In [7]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'sentence', 'year', 'label', 'orig_index', 'input_ids', 'attention_mask'],
        num_rows: 1984
    })
    test: Dataset({
        features: ['index', 'sentence', 'year', 'label', 'orig_index', 'input_ids', 'attention_mask'],
        num_rows: 496
    })
})

In [ ]:
from transformers import DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora-output",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=5e-5,
    fp16=True,
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    warmup_steps=50,  # Add warmup
    lr_scheduler_type="cosine",  # Add learning rate scheduler
    max_grad_norm=1.0,  # Add gradient clipping
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/hice1/kmadaswar3/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.411600
20,3.235400
30,3.226200
40,3.188700
50,3.007900
60,2.937100
70,2.813000
80,2.752400
90,2.754600
100,2.707600


/home/hice1/kmadaswar3/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/hice1/kmadaswar3/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return 

In [ ]:
# Save the LoRA adapters (only saves the small adapter weights)
model.save_pretrained("./model-update-1")
tokenizer.save_pretrained("./model-update-1")

('./model-update-1/tokenizer_config.json',
 './model-update-1/special_tokens_map.json',
 './model-update-1/tokenizer.model',
 './model-update-1/added_tokens.json')